# Unit 02 · Shape and metrics

From a CT's pelvis mask to **the 64³ cube the network will predict**, a mesh you can look at, and the
numbers that say how good a reconstruction is. Then the dataset every later unit trains and tests on.

**Needs:** unit 01 moved into `src/xray3d` (`volume.py`, `physics.py`, `geometry.py`). The dataset build
(part C) also calls `touches_scan_edge`, `hu_to_mu` and `project_pinhole`.

| Part | After video | What you build | Moves into | Suggested driver |
|---|---|---|---|---|
| A | 05 · Representing 3D shape | the canonical cube: where it goes, resampling, occupancy, marching cubes | `shape.py` | Adir |
| B | 06 · Measuring a reconstruction | Dice, IoU, surface distances, HD95, ASSD, NSD, Chamfer, a clinical angle | `metrics.py` | Noa |
| C | – | the processed dataset (Claude's `build_dataset.ipynb`) and its geometry check | – | Adir |

Within a part, swap the keyboard on every TODO: one types, the other explains. Record it in `LOG.md`.

In [ ]:
# --- setup (written by Claude, no need to change it): the same cell works locally, on Colab and on Kaggle ---
import os, sys, subprocess
from pathlib import Path

UNIT = "02-shape-and-metrics"
REPO_URL = "https://github.com/AdirBuskila/pelvis-xray-3d"
IN_CLOUD = "google.colab" in sys.modules or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
if IN_CLOUD:
    root = Path("/content" if "google.colab" in sys.modules else "/kaggle/working") / "pelvis-xray-3d"
    if not root.exists():  # Kaggle: turn Internet on in the notebook settings first
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)], check=True)
else:  # local: the repo is the first folder above this notebook that has a pyproject.toml
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root / "units" / UNIT)
for p in (root / "src", Path.cwd()):  # import xray3d (even before `pip install -e .`) and this unit's checks
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import xray3d
from xray3d import data, env, plotting
import test_unit as T

env.summary()

In [ ]:
from xray3d import volume, physics, geometry, prep
missing = [f"{m.__name__}.{f}" for m, f in [(volume, "voxel_spacing"), (volume, "voxel_to_mm"), (volume, "mask_volume_cm3"), (volume, "touches_scan_edge"),
                                            (physics, "hu_to_mu"), (geometry, "project_pinhole")] if not hasattr(m, f)]
print("unit 01 functions missing from src/xray3d:", missing or "none, good")
if data.HERO not in data.list_subjects() and env.where() != "local":
    data.fetch_subjects(data.GOOD_SUBJECTS[:3])   # cloud: download 3 subjects (~150 MB) out of the remote zip

---
## Part A · The canonical cube (after video 05)

Video 05 showed four ways to hold a shape: voxels, points, meshes, implicit fields. The network predicts
**occupancy on a voxel grid**, so every pelvis must live on *the same* grid: 64³ voxels over a **fixed 320 mm
cube** (5 mm voxels), centred on the pelvis, with array axes in **RAS** order (index i → patient's Right,
j → Anterior, k → Superior). Fixed size, not scaled per patient: the network should see real sizes.

In [ ]:
hu, affine = data.load_ct(data.HERO)
masks = data.load_mask(data.HERO, data.PELVIS + data.FEMURS)
pelvis = masks["hip_left"] | masks["hip_right"] | masks["sacrum"]
spacing = volume.voxel_spacing(affine)
idx = np.argwhere(pelvis)
print("CT", hu.shape, "voxels of", spacing, "mm, axes", data.axcodes(affine))
print("pelvis bounding box (mm):", (idx.max(0) - idx.min(0) + 1) * spacing, " <- must fit in 320 mm")

### TODO A1 · `bbox_center_mm`  ·  suggested driver: **Adir**

Where the cube goes: the centre of the pelvis's **bounding box**, in world mm.

> **Nudge:** find the lowest and highest voxel index along each axis; the middle of those is a voxel index (maybe a half-integer). Which unit-01 tool turns an index into mm?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def bbox_center_mm(mask, affine):
    """World position (mm) of the centre of the mask's bounding box.

    mask    : (X, Y, Z) bool
    affine  : (4, 4) voxel-index -> mm
    returns : (3,) float
    """
    raise NotImplementedError("TODO A1")

In [ ]:
T.check_bbox_center_mm(bbox_center_mm)

In [ ]:
centre = bbox_center_mm(pelvis, affine)
centroid = volume.voxel_to_mm(affine, idx.mean(axis=0)[None])[0]
print("bounding-box centre:", np.round(centre, 1), "mm   centroid:", np.round(centroid, 1), "mm")

**Explain-back:** the centroid is a few mm away from the box centre. Why does the project use the box
centre? (Hint: s1390's pelvis is 312 mm wide in a 320 mm cube. What has to be true about *both* ends?)

### TODO A2 · `cube_grid`  ·  suggested driver: **Noa**

The **centre of every voxel** of the cube, in world mm: an array of shape `(n, n, n, 3)` where
`grid[i, j, k]` is the (x, y, z) of voxel (i, j, k). The same function makes the bigger CT box for DiffDRR
in part C, so `size_mm` and `n` may also be 3-tuples (one per axis).

> **Nudge:** along one axis, the cube runs from centre − size/2 to centre + size/2 in n cells. Where is the middle of cell 0? Make the three 1D axes first; `np.meshgrid(..., indexing="ij")` builds the 3D grid.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def cube_grid(center_mm, size_mm=320.0, n=64):
    """Voxel centres (mm) of a box centred at center_mm, axes aligned with the world (RAS) axes.

    center_mm : (3,)
    size_mm   : box side in mm, a number or (3,) per axis
    n         : voxels per side, a number or (3,) per axis
    returns   : (n0, n1, n2, 3), grid[i, j, k] = (x, y, z) of voxel (i, j, k)
    """
    raise NotImplementedError("TODO A2")

In [ ]:
T.check_cube_grid(cube_grid)

### TODO A3 · `resample`  ·  suggested driver: **Adir**

Read a volume **at arbitrary world points**. A CT stores values at its own voxels; the cube's voxel centres
fall between them. Go world → voxel index with the **inverse affine**, then let
`scipy.ndimage.map_coordinates(vol, ijk.T, order=order, mode="constant", cval=fill)` interpolate
(that call is plumbing: the concept is which index you hand it, which order, and what lies outside).

> **Nudge:** unit 01's voxel_to_mm went index → mm with the affine. Which matrix goes back? map_coordinates wants the indices as (3, N), so flatten the points first and reshape at the end.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
from scipy import ndimage

def resample(vol, affine, points_mm, order=1, fill=0.0):
    """Values of `vol` at world points.

    vol       : (X, Y, Z) array on the grid described by `affine`
    points_mm : (..., 3) world positions (e.g. a whole cube_grid)
    order     : 0 = nearest voxel, 1 = linear interpolation
    fill      : value for points outside the volume (e.g. -1000 HU = air for a CT)
    returns   : (...) values, the points' shape without the last axis
    """
    raise NotImplementedError("TODO A3")

In [ ]:
T.check_resample(resample)

In [ ]:
grid = cube_grid(centre)                                   # (64, 64, 64, 3)
j = 32                                                     # a coronal layer through the middle of the cube
ct_lin = resample(hu, affine, grid[:, j], order=1, fill=-1000)
ct_near = resample(hu, affine, grid[:, j], order=0, fill=-1000)
j_full = int(round(np.linalg.solve(affine, np.r_[centre, 1])[1]))          # the same plane in the full CT
full = hu[:, j_full, :]
show = lambda a: np.clip(a, -200, 1200).T[::-1, ::-1]       # rows head -> feet, columns patient's Right -> Left
plotting.show_row([show(full), show(ct_near), show(ct_lin)],
                  ["full CT, 1.5 mm", "cube, 5 mm, order=0", "cube, 5 mm, order=1"], size=4.5)

**Answer together:** the full slice is wider than the cube: what got cut, and why is that fine? Where in the
5 mm images do you see the difference between order 0 and 1? For the CT we use linear; what would linear do
to a **label map** with values 0, 1, 2 (hip, sacrum, ...)?

### TODO A4 · `occupancy`  ·  suggested driver: **Noa**

The pelvis on the cube: resample the mask **as a float** (linear), then threshold at 0.5 (`>= 0.5`).
Return `uint8` 0/1 (the dataset's format).

> **Nudge:** a value of 0.7 at a cube voxel means: 70 % of the way from 'outside' to 'inside' voxels. Your A2 and A3 do almost all the work; what must happen to a bool mask before linear interpolation makes sense?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def occupancy(mask, affine, center_mm, size_mm=320.0, n=64):
    """Occupancy of `mask` on the canonical cube: (n, n, n) uint8, 1 = inside."""
    raise NotImplementedError("TODO A4")

In [ ]:
T.check_occupancy(occupancy)

In [ ]:
occ = occupancy(pelvis, affine, centre)
cm3 = volume.mask_volume_cm3(pelvis, spacing)
print(f"{int(occ.sum())} occupied voxels = {occ.sum() * 0.125:.0f} cm^3   (the 1.5 mm mask: {cm3:.0f} cm^3)")
views = [occ.max(axis=1).T[::-1, ::-1], occ.max(axis=0).T[::-1, ::-1], occ.max(axis=2).T[::-1, ::-1]]
plotting.show_row(views, ["max along A (front view)", "max along R (side view, A on the left)", "max along S (from above)"],
                  cmap="bone", size=4)

**Resolution is a choice.** The same pelvis at 32³ (10 mm), 64³ (5 mm) and 128³ (2.5 mm): memory grows as n³.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 4.5))
for a, n in zip(ax, [32, 64, 128]):
    o = occupancy(pelvis, affine, centre, 320.0, n)
    a.imshow(prep.coronal(o), cmap="bone", interpolation="nearest")
    a.set_title(f"{n}^3: {320 / n:.1f} mm voxels, {n ** 3 * 4 / 2 ** 20:.1f} MB as float32\n"
                f"volume {o.sum() * (320 / n) ** 3 / 1000:.0f} cm^3")
    a.axis("off")
plt.tight_layout(); plt.show()

**Explain-back:** at 32³ the thin parts (the iliac wings, the pubic rami) break up. Why exactly there, and
what is the smallest bone thickness 64³ can show? Why didn't we take 128³ (think of a batch of 8 in a 3D
decoder on a free GPU)?

### TODO A5 · `mesh_from_occupancy`  ·  suggested driver: **Adir**

**Marching cubes** turns the occupancy into a triangle mesh: `skimage.measure.marching_cubes(vol, 0.5,
spacing=(cell_mm,) * 3)` returns vertices measured in mm **from voxel (0, 0, 0)'s centre** (plumbing). Your
part: put the vertices into the **world** (or cube) frame, and make sure the surface is closed even where the
bone touches the edge of the array. An empty cube has no surface: return empty arrays.

> **Nudge:** where is voxel (0,0,0)'s centre relative to center_mm (part A2)? A surface needs an outside on both sides: what can you add around the array so every voxel at its edge has empty neighbours, and how does that shift the vertices?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
from skimage import measure

def mesh_from_occupancy(occ, cell_mm=5.0, center_mm=(0.0, 0.0, 0.0)):
    """Triangle mesh of the 0.5 level of an occupancy cube.

    occ       : (n, n, n) 0/1
    center_mm : where the cube's centre sits (default: the cube frame, origin at its centre)
    returns   : verts (V, 3) in mm, faces (F, 3) int indices into verts. Empty cube -> (0, 3), (0, 3).
    """
    raise NotImplementedError("TODO A5")

In [ ]:
T.check_mesh_from_occupancy(mesh_from_occupancy)

In [ ]:
verts, faces = mesh_from_occupancy(occ, 5.0, centre)
print(f"{len(verts)} vertices, {len(faces)} triangles")
path = prep.write_obj(env.data_root() / "derived" / f"{data.HERO}_pelvis64.obj", verts, faces)
print("saved", path, "(open it in Windows 3D Viewer / Blender / MeshLab)")
fig = plt.figure(figsize=(12, 6))
prep.show_mesh(verts - centre, faces, "s1390, 64^3, from the front", ax=fig.add_subplot(1, 2, 1, projection="3d"), view=(5, 90))
prep.show_mesh(verts - centre, faces, "from the side", ax=fig.add_subplot(1, 2, 2, projection="3d"), view=(5, 180))
plt.show()

**Explain-back:** where does the mesh's staircase look come from, and why would smoothing the occupancy (or
predicting probabilities) give a smoother mesh? Look up the sign: did the mesh's front face the viewer
(y = Anterior) in the left picture?

**Part A done.** Move `bbox_center_mm`, `cube_grid`, `resample`, `occupancy` and `mesh_from_occupancy` into
`src/xray3d/shape.py` (with `from scipy import ndimage` / `from skimage import measure` inside or at the top).
Claude's dataset build (part C) uses exactly these.

---
## Part B · Measuring a reconstruction (after video 06)

We need "predictions" with known mistakes. Claude made them from real masks, on your cube:

* **gt**: s1390's pelvis; **other**: another person's pelvis (s1361), centred the same way;
* **wing**: the top 75 mm of the left iliac wing removed; **shift**: gt moved 2 voxels (10 mm) towards the head;
* **empty**: nothing predicted.

In [ ]:
cases = {"gt": occ}
m2 = data.load_mask("s1361", data.PELVIS)
p2 = m2["hip_left"] | m2["hip_right"] | m2["sacrum"]
a2 = data.load_nifti(data.mask_path("s1361", "sacrum")).affine
cases["other"] = occupancy(p2, a2, bbox_center_mm(p2, a2))
hl = masks["hip_left"].copy()
hl[:, :, np.argwhere(hl)[:, 2].max() - int(75 / spacing[2]):] = False        # cut the top 75 mm (axis 2 = S here)
cases["wing"] = occupancy(hl | masks["hip_right"] | masks["sacrum"], affine, centre)
cases["shift"] = np.roll(occ, 2, axis=2)
cases["empty"] = np.zeros_like(occ)
j = int(np.argmax(occ.sum(axis=(0, 2))))
fig, ax = plt.subplots(1, 4, figsize=(16, 4.5))
for a, k in zip(ax, ["other", "wing", "shift", "empty"]):
    a.imshow(prep.overlap_image(prep.coronal(occ, j), prep.coronal(cases[k], j)), interpolation="nearest")
    a.set_title(k); a.axis("off")
fig.suptitle("coronal layer of gt vs each case: white = both, red = extra (FP), blue = missed (FN)")
plt.tight_layout(); plt.show()

### TODO B1 · `dice`  ·  suggested driver: **Noa**

**Dice** = 2|A∩B| / (|A| + |B|). Inputs are 0/1 arrays (any dtype). Two empty masks agree perfectly (1.0).

> **Nudge:** |A∩B| is a count of voxels that are 1 in both. How do you get that from two 0/1 arrays without a loop, and what if |A| + |B| is 0?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def dice(a, b):
    """Dice overlap of two binary masks, in [0, 1]. Both empty -> 1.0."""
    raise NotImplementedError("TODO B1")

In [ ]:
T.check_dice(dice)

### TODO B2 · `iou`  ·  suggested driver: **Adir**

**IoU** (Jaccard) = |A∩B| / |A∪B|. Both empty → 1.0.

> **Nudge:** same counting as Dice, a different denominator. Can you also write IoU from Dice alone?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def iou(a, b):
    """Intersection over union of two binary masks, in [0, 1]. Both empty -> 1.0."""
    raise NotImplementedError("TODO B2")

In [ ]:
T.check_iou(iou)

In [ ]:
for k, v in cases.items():
    print(f"{k:<6} Dice {dice(occ, v):.3f}   IoU {iou(occ, v):.3f}")

**Answer together:** "wing" lost a whole piece of bone, yet its Dice is the best of the imperfect ones. Why?
Overlap counts voxels; most voxels of a pelvis are far from the missing wing. The next metrics look at
**surfaces** instead.

### TODO B3 · `surface_voxels`  ·  suggested driver: **Noa**

The **surface** of a mask: its voxels that have at least one empty neighbour (6-neighbourhood: ±1 along one
axis). Outside the array counts as empty. `scipy.ndimage.binary_erosion` removes exactly the voxels that have
an empty neighbour.

> **Nudge:** erosion keeps the voxels whose neighbours are ALL inside. The surface is what erosion took away: which boolean operation between the mask and its erosion gives that?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def surface_voxels(mask):
    """Bool mask of the surface voxels of `mask` (same shape)."""
    raise NotImplementedError("TODO B3")

In [ ]:
T.check_surface_voxels(surface_voxels)

### TODO B4 · `surface_distances`  ·  suggested driver: **Adir**

For every surface voxel of `a`, the distance (mm) to the **nearest surface voxel of `b`**. Doing it with a
loop over pairs is 10⁴ × 10⁴ distances. Instead: a **Euclidean distance transform**
(`ndimage.distance_transform_edt(x, sampling=spacing_mm)`) gives, for every voxel where `x` is True, the
distance to the nearest voxel where `x` is False. If `b` has no surface, every distance is `inf`.

> **Nudge:** you want 'distance to the nearest surface voxel of b' everywhere. What array do you hand the EDT so that b's surface voxels are its zeros? Then read that map only at a's surface voxels.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def surface_distances(a, b, spacing_mm=5.0):
    """1D array: for each surface voxel of a, the distance (mm) to the nearest surface voxel of b.

    spacing_mm : voxel size, a number or (3,) per axis
    """
    raise NotImplementedError("TODO B4")

In [ ]:
T.check_surface_distances(surface_distances)

In [ ]:
d_go = surface_distances(occ, cases["other"])     # gt surface -> other's surface
d_og = surface_distances(cases["other"], occ)     # and back
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist([d_go, d_og], bins=np.arange(0, 45, 2.5), label=["gt -> other", "other -> gt"], color=["#4a90c2", "#e0a040"])
ax.set_xlabel("distance to the other surface (mm)"); ax.set_ylabel("surface voxels"); ax.legend(); plt.show()
print(f"max {max(d_go.max(), d_og.max()):.1f} mm, 95th percentiles {np.percentile(d_go, 95):.1f} / {np.percentile(d_og, 95):.1f} mm,"
      f" means {d_go.mean():.1f} / {d_og.mean():.1f} mm")

The three summaries of these distances (video 06): the **worst** (Hausdorff), the **95th percentile** (HD95:
the worst, but ignoring the 5 % most extreme surface voxels), and the **mean** (ASSD). Each is computed over
**both directions**: a prediction that's a tiny ball inside the pelvis has all its own surface close to gt's,
but gt's surface is far from it.

### TODO B5 · `hd95`  ·  suggested driver: **Noa**

**HD95** (mm): the larger of the two directed 95th percentiles, `np.percentile(d, 95)` (numpy's default
interpolation, the check uses it). Both empty → 0.0; exactly one empty → `inf`.

> **Nudge:** two calls of your B4 (which ways round?), a percentile each, then which of the two do you report?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def hd95(gt, pred, spacing_mm=5.0):
    """95th-percentile symmetric Hausdorff distance (mm). Both empty -> 0.0, one empty -> inf."""
    raise NotImplementedError("TODO B5")

In [ ]:
T.check_hd95(hd95)

### TODO B6 · `assd`  ·  suggested driver: **Adir**

**ASSD** (mm): the average symmetric surface distance: the mean over **all** surface voxels of both shapes
(pool the two lists, then average). Both empty → 0.0; one empty → `inf`.

> **Nudge:** pooling isn't the same as averaging the two means when the surfaces have different sizes. The check has a case where it matters.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def assd(gt, pred, spacing_mm=5.0):
    """Average symmetric surface distance (mm). Both empty -> 0.0, one empty -> inf."""
    raise NotImplementedError("TODO B6")

In [ ]:
T.check_assd(assd)

### TODO B7 · `nsd`  ·  suggested driver: **Noa**

**NSD** (normalised surface distance, a.k.a. surface Dice) at tolerance τ: the fraction of surface voxels
(both shapes, pooled) that lie **within τ mm** (≤ τ) of the other surface. 1 is perfect. Both empty → 1.0;
one empty → 0.0. The proposal says τ = 1.5 mm; we report τ = 5 mm too (see the explainer).

> **Nudge:** same two distance lists as B5/B6; this time count instead of summarise.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def nsd(gt, pred, tau_mm, spacing_mm=5.0):
    """Fraction of surface voxels of both shapes within tau_mm of the other surface. Both empty -> 1, one empty -> 0."""
    raise NotImplementedError("TODO B7")

In [ ]:
T.check_nsd(nsd)

In [ ]:
print(f"{'case':<6} {'Dice':>6} {'HD95':>7} {'ASSD':>6} {'NSD1.5':>7} {'NSD5':>6}")
for k, v in cases.items():
    print(f"{k:<6} {dice(occ, v):6.3f} {hd95(occ, v):7.1f} {assd(occ, v):6.2f} {nsd(occ, v, 1.5):7.3f} {nsd(occ, v, 5.0):6.3f}")

**Answer together:** order the cases by Dice, then by HD95. Which metric "notices" the missing wing, and
which one notices the shift? Why is NSD at 1.5 mm almost the same number as "the fraction of surface voxels
that are exactly on the other surface"? (What distances can two 5 mm voxel grids even produce?)

### TODO B8 · `chamfer`  ·  suggested driver: **Adir**

**Chamfer distance** between two point clouds (e.g. mesh vertices): for each point, the distance to the
nearest point of the other cloud; average each direction, then average the two means (mm, not squared).
Finding nearest neighbours is plumbing: `scipy.spatial.cKDTree(b).query(a)` returns `(distances, indices)`.
Either cloud empty → `inf` (both empty → 0.0).

> **Nudge:** it's the same idea as B4 + B6, but on points instead of voxels. Which tree answers 'nearest point of b' for every point of a?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
from scipy.spatial import cKDTree

def chamfer(points_a, points_b):
    """Symmetric Chamfer distance (mm): (mean NN distance a->b + mean NN distance b->a) / 2."""
    raise NotImplementedError("TODO B8")

In [ ]:
T.check_chamfer(chamfer)

In [ ]:
mesh = {k: mesh_from_occupancy(v)[0] for k, v in cases.items()}     # vertices in the cube frame
for k in cases:
    print(f"{k:<6} Chamfer {chamfer(mesh['gt'], mesh[k]):6.2f} mm   ({len(mesh[k])} vertices)")

### A clinical angle: the centre-edge (CE) angle, simplified

Dice says nothing a surgeon would use. The **CE angle** (Wiberg) is: on a coronal slice through the centre of
the femoral head, the angle between the **vertical** and the line from the head centre to the **lateral edge of
the acetabular roof** (the hip socket's rim). Normal adults: about 25–40°; small = a shallow socket.

The head belongs to the femur, which we don't predict, so its centre and radius come from the ground-truth femur
(Claude's `prep.femoral_head`, a distance transform: the deepest point of the femur's top). The roof comes from
the shape being measured: gt or a prediction.

In [ ]:
heads = {}
for side in ("left", "right"):
    c_mm, r = prep.femoral_head(masks[f"femur_{side}"], affine)
    heads[side] = (c_mm - centre, r)                     # in the cube frame (origin at the cube centre)
    print(f"{side:<5} femoral head: centre {np.round(c_mm - centre, 1)} mm, radius {r:.1f} mm")

### TODO B9 · `ce_angle`  ·  suggested driver: **Noa**

The simplified CE angle on a cube (degrees). Take the occupied voxels (their centres: your `cube_grid` around
the origin) in the **coronal layer through the head** (|y − y_head| ≤ cell/2), **above** the head centre
(z > z_head) and in a **band around the head** (distance from the centre between r − cell/2 and r + 1.5 cell).
For each, the angle from the vertical towards **lateral** (away from the midline: −x for the left hip, +x for
the right) is `atan2(lateral offset, upward offset)`. The CE angle is the largest. No such voxels → `nan`.

> **Nudge:** make everything relative to the head centre first; then 'lateral offset' is just −dx or +dx depending on the side. Filter with a boolean mask, then np.degrees(np.arctan2(...)).max().

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def ce_angle(occ, head_mm, radius_mm, side, cell_mm=5.0):
    """Simplified centre-edge angle (degrees) of one hip on an occupancy cube (cube frame, mm).

    head_mm   : (3,) femoral head centre in the cube frame;  radius_mm: its radius
    side      : "left" or "right" (the patient's)
    returns   : float degrees, nan if no acetabular roof voxel is found
    """
    raise NotImplementedError("TODO B9")

In [ ]:
T.check_ce_angle(ce_angle)

In [ ]:
for k in ["gt", "wing", "shift"]:                      # (the heads are s1390's, so "other" makes no sense here)
    print(f"{k:<6} CE left {ce_angle(cases[k], *heads['left'], 'left'):5.1f} deg   right {ce_angle(cases[k], *heads['right'], 'right'):5.1f} deg")

**Explain-back:** "wing" has the better Dice and the much worse HD95, yet its CE angle is untouched; "shift" ruins
the angle (the roof moved 10 mm away from the head). Which metric would have warned you about each? On a 5 mm grid, one voxel at the rim of a 25 mm head changes the angle by about atan(5/25) ≈ 11°: what does
that say about how precisely we can measure it on 64³? (Unit 07 plots Dice against this angle.)

### TODO B10 · `evaluate`  ·  suggested driver: **Adir**

One function that scores a prediction the way units 03–08 will report it: a dict with exactly the keys
`dice, iou, hd95, assd, nsd_1.5mm, nsd_5mm, chamfer` (Chamfer on the vertices of your A5 meshes).

> **Nudge:** it's all calls of your own functions. What does mesh_from_occupancy return for an empty prediction, and what does your chamfer do with it?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def evaluate(gt, pred, cell_mm=5.0):
    """All metrics of one prediction against its ground truth (both (n, n, n) 0/1 cubes)."""
    raise NotImplementedError("TODO B10")

In [ ]:
T.check_evaluate(evaluate)

In [ ]:
rows = {k: evaluate(occ, v) for k, v in cases.items()}
print(f"{'case':<6}" + "".join(f"{c:>11}" for c in T.KEYS_ORDER))
for k, m in rows.items():
    print(f"{k:<6}" + "".join(f"{m[c]:11.3f}" for c in T.KEYS_ORDER))

**Sanity tests (the proposal's week 2):** gt vs gt must be perfect on every metric, empty vs gt the worst
possible. Check the first and last rows. Then: which single number would you put in the abstract, and which two
would you put next to it so a reader can't be fooled?

**Part B done.** Move `dice`, `iou`, `surface_voxels`, `surface_distances`, `hd95`, `assd`, `nsd`, `chamfer`,
`ce_angle` and `evaluate` into `src/xray3d/metrics.py` (with their imports: `ce_angle` and `evaluate` call your
part-A functions, so add `from xray3d import shape` and write `shape.cube_grid(...)`, `shape.mesh_from_occupancy(...)`). Units 03–08 import them from there.

---
## Part C · The processed dataset

Claude's `build_dataset.ipynb` (in this folder) runs **your** functions over every usable subject: screening
(your `touches_scan_edge` + the metal rule), the cube (your `bbox_center_mm`, `occupancy`), and the two X-rays,
rendered by DiffDRR from the CT resampled with your `cube_grid` + `resample` and turned into μ by your
`hu_to_mu`. Locally that's the 27 small-subset pelvises; on Kaggle the full dataset (~300 expected).

The X-ray geometry is fixed for the whole project (`dataset.geometry()`): source 750 mm from the cube centre,
detector 1,020 mm from the source, 128 × 128 pixels of 4 mm. `dataset.camera(view)` gives `K, R, t` in the
cube frame, so **your** `project_pinhole` says where any point of the cube lands in each image.

In [ ]:
from xray3d import dataset
if not dataset.exists():
    print("No processed dataset here yet: run build_dataset.ipynb (or, on Colab/Kaggle, dataset.ensure_local()).")
else:
    ix = dataset.load_index()
    print(len(ix["subjects"]), "subjects, built", ix["created"], "from", ix["source"])
    print("in the split:", {k: len(dataset.subjects(k)) for k in ("train", "val", "test")})
    d = dataset.load_subject(data.HERO if data.HERO in ix["subjects"] else ix["subjects"][0])
    plotting.show_row([d["ap"], d["lat"], d["ap_bone"], d["lat_bone"]], ["ap", "lat", "ap_bone", "lat_bone"], size=4)

**The geometry check.** If the cameras are right, projecting the occupied voxels with your `project_pinhole`
must cover the same pixels as the pelvis-only rendering (`*_bone > 0`). The acceptance bar is IoU ≥ 0.9.

In [ ]:
if dataset.exists():
    ious = {v: [] for v in dataset.VIEWS}
    for sid in dataset.subjects()[:6]:
        dd = dataset.load_subject(sid)
        for v in dataset.VIEWS:
            ious[v].append(prep.silhouette_iou(dd, v, geometry.project_pinhole))
    for v in dataset.VIEWS:
        print(f"{v:<3} silhouette IoU over {len(ious[v])} subjects: min {min(ious[v]):.3f}, mean {np.mean(ious[v]):.3f}")
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    for a, v in zip(ax, dataset.VIEWS):
        a.imshow(prep.overlap_image(d[f"{v}_bone"] > 1e-3, prep.project_occupancy(d["occ"], v, geometry.project_pinhole)))
        a.set_title(v); a.axis("off")
    fig.suptitle("white = both, red = projected voxels only, blue = rendered pelvis only")
    plt.tight_layout(); plt.show()
    assert min(min(x) for x in ious.values()) >= 0.9, "silhouette IoU below 0.9: the cameras and the images disagree"

In [ ]:
try:
    import diffdrr  # noqa: F401
except ImportError:
    print("DiffDRR isn't installed here (on Colab: !pip install -q diffdrr); skipping the marker test.")
else:
    print("bright single voxels at known positions, rendered by DiffDRR vs projected by your project_pinhole:")
    print(f"largest error: {prep.check_geometry(geometry.project_pinhole):.2f} px")

**Explain-back:** the IoU isn't 1.0 even though the geometry is exact (the marker test agrees to a fraction of a
pixel). Where do the red and blue pixels come from? (Compare a 5 mm voxel with a pixel at the isocentre.)
Why was the marker test needed at all: what could have gone wrong between DiffDRR's world frame and ours?

---
## Wrap-up: run the checks against `src/xray3d`

Functions you haven't moved yet show up as **skipped**, moved ones as **passed**.

In [ ]:
!{sys.executable} -m pytest -q -rs .

Fill in `LOG.md`: driver per TODO, hints used, the explain-back answers and the numbers. Then ask Claude to
review the code like a TA, and to commit.